In [13]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
import pandas as pd

from src.content_based import ContentBasedRecommender
from src.evaluation import evaluate_recommender
from src.hibrid import HybridRecommender
from src.matrix_factorization import MatrixFactorization
from src.preprocessing import leave_k_last


In [14]:
input_dir = Path("../data")

train = pd.read_csv(input_dir / "processed/train.csv")
test = pd.read_csv(input_dir / "processed/test.csv")
movies = pd.read_csv(input_dir / "ml-latest-small/movies.csv")

In [15]:
train_fit, val_fit = leave_k_last(train, k=1)

mf = MatrixFactorization(train=train_fit, val=val_fit, random_state=142,
                         learning_rate=0.01, lm=0.01)
cb = ContentBasedRecommender()

mf.fit(1)
cb.fit(movies=movies, ratings=train)

[01/100] Train BPR Loss: 0.7037 | Val BPR Loss: 0.7192
[02/100] Train BPR Loss: 0.6963 | Val BPR Loss: 0.7227
[03/100] Train BPR Loss: 0.6898 | Val BPR Loss: 0.7098
[04/100] Train BPR Loss: 0.6786 | Val BPR Loss: 0.7039
[05/100] Train BPR Loss: 0.6549 | Val BPR Loss: 0.6907
[06/100] Train BPR Loss: 0.6089 | Val BPR Loss: 0.6680
[07/100] Train BPR Loss: 0.5439 | Val BPR Loss: 0.6134
[08/100] Train BPR Loss: 0.4792 | Val BPR Loss: 0.5792
[09/100] Train BPR Loss: 0.4258 | Val BPR Loss: 0.5296
[10/100] Train BPR Loss: 0.3884 | Val BPR Loss: 0.4918
[11/100] Train BPR Loss: 0.3613 | Val BPR Loss: 0.4795
[12/100] Train BPR Loss: 0.3408 | Val BPR Loss: 0.4441
[13/100] Train BPR Loss: 0.3258 | Val BPR Loss: 0.4125
[14/100] Train BPR Loss: 0.3136 | Val BPR Loss: 0.4007
[15/100] Train BPR Loss: 0.3026 | Val BPR Loss: 0.3959
[16/100] Train BPR Loss: 0.2928 | Val BPR Loss: 0.3708
[17/100] Train BPR Loss: 0.2874 | Val BPR Loss: 0.3906
[18/100] Train BPR Loss: 0.2812 | Val BPR Loss: 0.3410
[19/100] T

In [16]:
userId = 1

unique_items = train["movieId"].unique()
n_items = len(unique_items)
mf_scores = mf.recommend_top_n(userId, n_items, filtered_watched=False)
cb_scores = cb.recommend_top_n(userId, n_items)


mf_scores = mf.recommend_top_n(userId, n=len(mf.idx_to_item_id), filtered_watched=False)
cb_scores = cb.recommend_top_n(userId, n=len(movies), filtered_watched=False)

df_mf = pd.DataFrame(mf_scores, columns=['movieId', 'mf_score'])
df_cb = pd.DataFrame(cb_scores, columns=['movieId', 'cb_score'])

combined = pd.merge(df_mf, df_cb, on='movieId', how='outer')

In [17]:
print(combined.isna().sum())

movieId      0
mf_score    54
cb_score     0
dtype: int64


In [18]:
mf_score = combined["mf_score"]
cb_score = combined["cb_score"]

combined["mf_norm"] = (mf_score - mf_score.min()) / (mf_score.max() - mf_score.min())
combined["cb_norm"] = (cb_score - cb_score.min()) / (cb_score.max() - cb_score.min())

alpha = 0.5

weighted_score = alpha * combined["mf_norm"] + (1 - alpha) * combined["cb_norm"]

combined["final_score"] = weighted_score.fillna(combined["mf_norm"]).fillna(combined["cb_norm"])
recommendations = combined.sort_values(by="final_score", ascending=False)

In [19]:
print(recommendations.head(10))

      movieId  mf_score  cb_score   mf_norm   cb_norm  final_score
337       380  5.094841  0.717310  0.959868  0.892235     0.926051
126       153  4.651494  0.739373  0.906214  0.919679     0.912946
899      1197  4.781592  0.705089  0.921958  0.877035     0.899497
615       780  5.177695  0.663233  0.969895  0.824971     0.897433
418       480  5.055282  0.663233  0.955080  0.824971     0.890025
2014     2683  4.510774  0.713212  0.889184  0.887138     0.888161
224       260  5.298326  0.615892  0.984493  0.766085     0.875289
1154     1517  4.210248  0.713212  0.852815  0.887138     0.869976
898      1196  5.154672  0.615892  0.967108  0.766085     0.866597
1158     1527  4.060751  0.707531  0.834723  0.880072     0.857397


In [20]:
recommendations_with_titles = recommendations.merge(movies[["movieId", "title"]], 
                                                    on="movieId", 
                                                    how="left")

top_recommendations = recommendations_with_titles[["movieId", "title", "final_score"]].head(10)
print(top_recommendations)

   movieId                                              title  final_score
0      380                                   True Lies (1994)     0.926051
1      153                              Batman Forever (1995)     0.912946
2     1197                         Princess Bride, The (1987)     0.899497
3      780               Independence Day (a.k.a. ID4) (1996)     0.897433
4      480                               Jurassic Park (1993)     0.890025
5     2683       Austin Powers: The Spy Who Shagged Me (1999)     0.888161
6      260          Star Wars: Episode IV - A New Hope (1977)     0.875289
7     1517  Austin Powers: International Man of Mystery (1...     0.869976
8     1196  Star Wars: Episode V - The Empire Strikes Back...     0.866597
9     1527                          Fifth Element, The (1997)     0.857397


In [21]:
hybrid_recs = HybridRecommender(
    model_a=mf,
    model_b=cb,
    movies_df=movies,
    ratings_df=train,
    alpha=0.5
)

recs = hybrid_recs.recommend_top_n(user_id=1, n=10)
print(recs)

      movieId                                              title  final_score  \
314       380                                   True Lies (1994)     0.926051   
117       153                              Batman Forever (1995)     0.912946   
1837     2683       Austin Powers: The Spy Who Shagged Me (1999)     0.888161   
1059     1527                          Fifth Element, The (1997)     0.857397   
6          10                                   GoldenEye (1995)     0.846426   
1108     1610                   Hunt for Red October, The (1990)     0.839487   
4196     6539  Pirates of the Caribbean: The Curse of the Bla...     0.834245   
91        112         Rumble in the Bronx (Hont faan kui) (1995)     0.833619   
965      1374             Star Trek II: The Wrath of Khan (1982)     0.822859   
395       485                            Last Action Hero (1993)     0.819737   

        norm_a    norm_b  
314   0.959868  0.892235  
117   0.906214  0.919679  
1837  0.889184  0.887138  


In [ ]:

alphas = [round(x * 0.1, 1) for x in range(11)]
results = []
k_eval = 10


for alpha in alphas:

    hybrid = HybridRecommender(
        model_a=mf,
        model_b=cb,
        movies_df=movies,
        ratings_df=train,
        alpha=alpha
    )
    

    metrics = evaluate_recommender(hybrid, test, k=k_eval)
    

    metrics["alpha"] = alpha
    results.append(metrics)
    
    print(f"Alpha: {alpha:.1f} | Precision@{k_eval}: {metrics[f'Precision@{k_eval}']:.4f} | "
          f"Recall@{k_eval}: {metrics[f'Recall@{k_eval}']:.4f} | NDCG@{k_eval}: {metrics[f'NDCG@{k_eval}']:.4f}")


df_results = pd.DataFrame(results)


cols = ["alpha"] + [c for c in df_results.columns if c != "alpha"]
df_results = df_results[cols]


best_row = df_results.loc[df_results[f"NDCG@{k_eval}"].idxmax()]

print("="*50)
print(f"Оптимальный alpha по NDCG@{k_eval}: {best_row['alpha']}")
print("="*50)
print(df_results)

Alpha: 0.0 | Precision@10: 0.0005 | Recall@10: 0.0055 | NDCG@10: 0.0017
Alpha: 0.1 | Precision@10: 0.0008 | Recall@10: 0.0082 | NDCG@10: 0.0044
Alpha: 0.2 | Precision@10: 0.0011 | Recall@10: 0.0110 | NDCG@10: 0.0061
Alpha: 0.3 | Precision@10: 0.0022 | Recall@10: 0.0219 | NDCG@10: 0.0099
Alpha: 0.4 | Precision@10: 0.0027 | Recall@10: 0.0274 | NDCG@10: 0.0129
Alpha: 0.5 | Precision@10: 0.0033 | Recall@10: 0.0329 | NDCG@10: 0.0160
Alpha: 0.6 | Precision@10: 0.0041 | Recall@10: 0.0411 | NDCG@10: 0.0193
Alpha: 0.7 | Precision@10: 0.0058 | Recall@10: 0.0575 | NDCG@10: 0.0262
Alpha: 0.8 | Precision@10: 0.0071 | Recall@10: 0.0712 | NDCG@10: 0.0328
Alpha: 0.9 | Precision@10: 0.0079 | Recall@10: 0.0795 | NDCG@10: 0.0382
Alpha: 1.0 | Precision@10: 0.0077 | Recall@10: 0.0767 | NDCG@10: 0.0334

Оптимальный alpha по NDCG@10: 0.9
    alpha  Precision@10  Recall@10   NDCG@10
0     0.0      0.000548   0.005479  0.001729
1     0.1      0.000822   0.008219  0.004396
2     0.2      0.001096   0.010959  0.

In [24]:
hybrid_recs = HybridRecommender(
    model_a=mf,
    model_b=cb,
    movies_df=movies,
    ratings_df=train,
    alpha=0.9
)

mf_metrics = evaluate_recommender(mf, test, k=10)
cb_metrics = evaluate_recommender(cb, test, k=10)
hybrid_metrics = evaluate_recommender(hybrid_recs, test, k=10)

results = pd.DataFrame([mf_metrics, cb_metrics, hybrid_metrics], 
                       index=["Matrix Factorization", "Content-Based", "Hybrid (Weighted+Switching)"])
print(results)

                             Precision@10  Recall@10   NDCG@10
Matrix Factorization             0.007671   0.076712  0.033597
Content-Based                    0.000548   0.005479  0.001729
Hybrid (Weighted+Switching)      0.007945   0.079452  0.038193
